In [1]:
import yaml
from types import SimpleNamespace
from omegaconf.omegaconf import OmegaConf
import sys
sys.path.append('/home/ubuntu/haitong-south-2/RLinf')
from rlinf.models import get_model
from rlinf.config import SupportedModel

def dict_to_namespace(d):
    """Recursively converts a dictionary and its nested dictionaries to SimpleNamespace."""
    if not isinstance(d, dict):
        return d
    # Convert the current dictionary to a SimpleNamespace and recurse on its values
    return SimpleNamespace(**{k: dict_to_namespace(v) for k, v in d.items()})
config_path = '/home/ubuntu/haitong-south-2/RLinf/examples/embodiment/config/libero_spatial_ppo_openpi_quickstart.yaml'
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg = OmegaConf.create(cfg)
cfg.actor.model.model_type = SupportedModel.OPENPI
cfg.actor.model.precision = None
cfg.actor.model.is_lora = False
cfg.actor.model.num_action_chunks = 5
cfg.actor.model.action_dim = 7
cfg.actor.model.openpi.config_name = "pi0_libero"
cfg.actor.model.openpi.num_images_in_input = 2
cfg.actor.model.openpi.noise_level = 0.5
cfg.actor.model.openpi.action_chunk = cfg.actor.model.num_action_chunks
cfg.actor.model.openpi.num_steps = 4
cfg.actor.model.openpi.train_expert_only = True
cfg.actor.model.openpi.action_env_dim = 7
cfg.actor.model.openpi.noise_method = "flow_sde"
cfg.actor.model.openpi.add_value_head = cfg.actor.model.add_value_head
cfg.actor.model.openpi.detach_critic_input = True


/home/ubuntu/haitong-south-2/RLinf/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-21 17:23:19,732	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
model = get_model(cfg.actor.model)

INFO:datasets:PyTorch version 2.6.0 available.
INFO:datasets:Polars version 1.37.1 available.
INFO:datasets:TensorFlow version 2.20.0 available.
INFO:datasets:JAX version 0.5.3 available.
INFO:root:Loaded norm stats from /home/ubuntu/cache/models/RLinf-Pi0-LIBERO-Spatial-Object-Goal-SFT/physical-intelligence/libero
INFO:root:Loaded norm stats from /home/ubuntu/cache/models/RLinf-Pi0-LIBERO-Spatial-Object-Goal-SFT/physical-intelligence/libero


In [6]:
import torch
isinstance(model.paligemma_with_expert, torch.nn.Module)

True

In [24]:
# create dummy input
import torch
bs = 8
channels = 3
size = 224
token_max_length = 4
state_dim = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

images = [torch.randn(bs, channels, size, size, device=device)] * 3
img_masks = [torch.ones(bs, device=device)] * 3
lang_tokens = torch.randint(0, 10000, (bs, token_max_length), device=device)
lang_masks = torch.ones(bs, token_max_length, device=device)
state = torch.randn(bs, state_dim)




In [36]:
prefix_embs, prefix_pad_masks, prefix_att_masks = model.embed_prefix(
            images, img_masks, lang_tokens, lang_masks
        )

In [37]:
print(prefix_embs.shape)
print(prefix_pad_masks.shape)
print(prefix_att_masks.shape)

torch.Size([8, 816, 2048])
torch.Size([8, 816])
torch.Size([8, 816])


In [38]:
from openpi.models_pytorch.pi0_pytorch import make_att_2d_masks
prefix_embs, prefix_pad_masks, prefix_att_masks = model.embed_prefix(
    images, img_masks, lang_tokens, lang_masks
)
prefix_att_2d_masks = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
prefix_position_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1

In [42]:
prefix_embs.shape

torch.Size([8, 816, 2048])

In [47]:
prefix_att_2d_masks_4d = model._prepare_attention_masks_4d(prefix_att_2d_masks)
model.paligemma_with_expert.paligemma.language_model.config._attn_implementation = "eager"  # noqa: SLF001

(prefix_output, _), past_key_values = model.paligemma_with_expert.forward(
    attention_mask=prefix_att_2d_masks_4d,
    position_ids=prefix_position_ids,
    past_key_values=None,
    inputs_embeds=[prefix_embs, None],
    use_cache=True,
)

In [62]:
denoise_inds = torch.arange(5)
denoise_inds = denoise_inds[None].repeat(bs, 1)
actions_shape = (bs, model.config.action_horizon, model.config.action_dim)
noise = model.sample_noise(actions_shape, device)

x_t = noise
noise.shape

torch.Size([8, 50, 32])

In [66]:
idx = denoise_inds[0]
sample_mode = "train"
num_steps = 5
compute_values = True
idx

tensor([0, 1, 2, 3, 4])

In [67]:
x_t_mean, x_t_std, value_t = model.sample_mean_var_val(
                x_t,
                0,
                state,
                prefix_pad_masks,
                past_key_values,
                sample_mode,
                num_steps,
                compute_values,
            )

In [70]:
x_t_mean.shape

torch.Size([8, 50, 32])

In [69]:
x_t_mean.shape

torch.Size([8, 50, 32])

In [46]:
print(prefix_output.shape)
print(o2.shape)

torch.Size([8, 816, 2048])


AttributeError: 'NoneType' object has no attribute 'shape'

# Forward from observation dict

In [4]:
from openpi.models.model import Observation
import jax
from openpi.models_pytorch.pi0_pytorch import PI0Pytorch

In [ ]:
bs = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
observation = Observation.from_dict(
    {
        'image': {
            'base_0_rgb': torch.randn(bs, 3, 224, 224),
            'left_wrist_0_rgb': torch.randn(bs, 3, 224, 224),
            'right_wrist_0_rgb': torch.randn(bs, 3, 224, 224),
        },
        'image_mask': {
            'base_0_rgb': torch.ones(bs, dtype=torch.bool),
            'left_wrist_0_rgb': torch.ones(bs, dtype=torch.bool),
            'right_wrist_0_rgb': torch.ones(bs, dtype=torch.bool),
        },
        'state': torch.randn(bs, 32),
        'tokenized_prompt': torch.randint(0, 10000, (bs, 48), dtype=torch.long),
        'tokenized_prompt_mask': torch.ones(bs, 48, dtype=torch.bool),
    }
)
actions = torch.randn(bs, 50, 32, device=device)
observation = jax.tree.map(lambda x: x.to(device), observation)

# Input to PI0pytorch forward should be channel-first

In [10]:
observation_transposed = Observation.from_dict(
    {
        'image': {
            'base_0_rgb': torch.randn(bs, 224, 224, 3),
            'left_wrist_0_rgb': torch.randn(bs, 224, 224, 3),
            'right_wrist_0_rgb': torch.randn(bs, 224, 224, 3),
        },
        'image_mask': {
            'base_0_rgb': torch.ones(bs, dtype=torch.bool),
            'left_wrist_0_rgb': torch.ones(bs, dtype=torch.bool),
            'right_wrist_0_rgb': torch.ones(bs, dtype=torch.bool),
        },
        'state': torch.randn(bs, 32),
        'tokenized_prompt': torch.randint(0, 10000, (bs, 48), dtype=torch.long),
        'tokenized_prompt_mask': torch.ones(bs, 48, dtype=torch.bool),
    }
)
actions = torch.randn(bs, 50, 32, device=device)
observation_transposed = jax.tree.map(lambda x: x.to(device), observation_transposed)

In [11]:
images, img_masks, lang_tokens, lang_masks, state = model._preprocess_observation(observation_transposed, train=True)
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, images))
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, img_masks))
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, lang_tokens))
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, lang_masks))
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, state))


[torch.Size([8, 224, 224, 3]), torch.Size([8, 224, 224, 3]), torch.Size([8, 224, 224, 3])]
[torch.Size([8]), torch.Size([8]), torch.Size([8])]
torch.Size([8, 48])
torch.Size([8, 48])
torch.Size([8, 32])


In [12]:
images, img_masks, lang_tokens, lang_masks, state = model._preprocess_observation(observation, train=True)
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, images))
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, img_masks))
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, lang_tokens))
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, lang_masks))
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, state))


[torch.Size([8, 3, 224, 224]), torch.Size([8, 3, 224, 224]), torch.Size([8, 3, 224, 224])]
[torch.Size([8]), torch.Size([8]), torch.Size([8])]
torch.Size([8, 48])
torch.Size([8, 48])
torch.Size([8, 32])


Does not really transpose the observation?

In [ ]:
from typing import Sequence
from openpi.models_pytorch.preprocessing_pytorch import IMAGE_KEYS, IMAGE_RESOLUTION
from openpi.shared import image_tools


def preprocess_observation_pytorch_colab(
    observation,
    *,
    train: bool = False,
    image_keys: Sequence[str] = IMAGE_KEYS,
    image_resolution: tuple[int, int] = IMAGE_RESOLUTION,
):
    """Torch.compile-compatible version of preprocess_observation_pytorch with simplified type annotations.

    This function avoids complex type annotations that can cause torch.compile issues.
    """
    if not set(image_keys).issubset(observation.images):
        raise ValueError(f"images dict missing keys: expected {image_keys}, got {list(observation.images)}")

    batch_shape = observation.state.shape[:-1]
    print(batch_shape)

    out_images = {}
    for key in image_keys:
        image = observation.images[key]

        # TODO: This is a hack to handle both [B, C, H, W] and [B, H, W, C] formats
        # Handle both [B, C, H, W] and [B, H, W, C] formats
        is_channels_first = image.shape[1] == 3  # Check if channels are in dimension 1

        if is_channels_first:
            print("is_channels_first, permute img")
            # Convert [B, C, H, W] to [B, H, W, C] for processing
            print(image.shape)
            image = image.permute(0, 2, 3, 1)
            print(image.shape)

        if image.shape[1:3] != image_resolution:
            # logger.info(f"Resizing image {key} from {image.shape[1:3]} to {image_resolution}")
            image = image_tools.resize_with_pad_torch(image, *image_resolution)

        if train:
            # Convert from [-1, 1] to [0, 1] for PyTorch augmentations
            image = image / 2.0 + 0.5

            # Apply PyTorch-based augmentations
            if "wrist" not in key:
                # Geometric augmentations for non-wrist cameras
                height, width = image.shape[1:3]

                # Random crop and resize
                crop_height = int(height * 0.95)
                crop_width = int(width * 0.95)

                # Random crop
                max_h = height - crop_height
                max_w = width - crop_width
                if max_h > 0 and max_w > 0:
                    # Use tensor operations instead of .item() for torch.compile compatibility
                    start_h = torch.randint(0, max_h + 1, (1,), device=image.device)
                    start_w = torch.randint(0, max_w + 1, (1,), device=image.device)
                    image = image[:, start_h : start_h + crop_height, start_w : start_w + crop_width, :]

                # Resize back to original size
                image = torch.nn.functional.interpolate(
                    image.permute(0, 3, 1, 2),  # [b, h, w, c] -> [b, c, h, w]
                    size=(height, width),
                    mode="bilinear",
                    align_corners=False,
                ).permute(0, 2, 3, 1)  # [b, c, h, w] -> [b, h, w, c]

                # Random rotation (small angles)
                # Use tensor operations instead of .item() for torch.compile compatibility
                angle = torch.rand(1, device=image.device) * 10 - 5  # Random angle between -5 and 5 degrees
                if torch.abs(angle) > 0.1:  # Only rotate if angle is significant
                    # Convert to radians
                    angle_rad = angle * torch.pi / 180.0

                    # Create rotation matrix
                    cos_a = torch.cos(angle_rad)
                    sin_a = torch.sin(angle_rad)

                    # Apply rotation using grid_sample
                    grid_x = torch.linspace(-1, 1, width, device=image.device)
                    grid_y = torch.linspace(-1, 1, height, device=image.device)

                    # Create meshgrid
                    grid_y, grid_x = torch.meshgrid(grid_y, grid_x, indexing="ij")

                    # Expand to batch dimension
                    grid_x = grid_x.unsqueeze(0).expand(image.shape[0], -1, -1)
                    grid_y = grid_y.unsqueeze(0).expand(image.shape[0], -1, -1)

                    # Apply rotation transformation
                    grid_x_rot = grid_x * cos_a - grid_y * sin_a
                    grid_y_rot = grid_x * sin_a + grid_y * cos_a

                    # Stack and reshape for grid_sample
                    grid = torch.stack([grid_x_rot, grid_y_rot], dim=-1)

                    image = torch.nn.functional.grid_sample(
                        image.permute(0, 3, 1, 2),  # [b, h, w, c] -> [b, c, h, w]
                        grid,
                        mode="bilinear",
                        padding_mode="zeros",
                        align_corners=False,
                    ).permute(0, 2, 3, 1)  # [b, c, h, w] -> [b, h, w, c]

            # Color augmentations for all cameras
            # Random brightness
            # Use tensor operations instead of .item() for torch.compile compatibility
            brightness_factor = 0.7 + torch.rand(1, device=image.device) * 0.6  # Random factor between 0.7 and 1.3
            image = image * brightness_factor

            # Random contrast
            # Use tensor operations instead of .item() for torch.compile compatibility
            contrast_factor = 0.6 + torch.rand(1, device=image.device) * 0.8  # Random factor between 0.6 and 1.4
            mean = image.mean(dim=[1, 2, 3], keepdim=True)
            image = (image - mean) * contrast_factor + mean

            # Random saturation (convert to HSV, modify S, convert back)
            # For simplicity, we'll just apply a random scaling to the color channels
            # Use tensor operations instead of .item() for torch.compile compatibility
            saturation_factor = 0.5 + torch.rand(1, device=image.device) * 1.0  # Random factor between 0.5 and 1.5
            gray = image.mean(dim=-1, keepdim=True)
            image = gray + (image - gray) * saturation_factor

            # Clamp values to [0, 1]
            image = torch.clamp(image, 0, 1)

            # Back to [-1, 1]
            image = image * 2.0 - 1.0

        # Convert back to [B, C, H, W] format if it was originally channels-first
        if is_channels_first:
            print("is_channels_first, permute img back")
            image = image.permute(0, 3, 1, 2)  # [B, H, W, C] -> [B, C, H, W]

        out_images[key] = image

    # obtain mask
    out_masks = {}
    for key in out_images:
        if key not in observation.image_masks:
            # do not mask by default
            out_masks[key] = torch.ones(batch_shape, dtype=torch.bool, device=observation.state.device)
        else:
            out_masks[key] = observation.image_masks[key]

    # Create a simple object with the required attributes instead of using the complex Observation class
    class SimpleProcessedObservation:
        def __init__(self, **kwargs):
            for key, value in kwargs.items():
                setattr(self, key, value)

    return SimpleProcessedObservation(
        images=out_images,
        image_masks=out_masks,
        state=observation.state,
        tokenized_prompt=observation.tokenized_prompt,
        tokenized_prompt_mask=observation.tokenized_prompt_mask,
        token_ar_mask=observation.token_ar_mask,
        token_loss_mask=observation.token_loss_mask,
    )

In [29]:
from openpi.models_pytorch.preprocessing_pytorch import preprocess_observation_pytorch
observation_processed_raw = preprocess_observation_pytorch_colab(observation, train=False)
print(jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, observation_processed_raw.images))

torch.Size([8])
is_channels_first, permute img
torch.Size([8, 3, 224, 224])
torch.Size([8, 224, 224, 3])
is_channels_first, permute img back
is_channels_first, permute img
torch.Size([8, 3, 224, 224])
torch.Size([8, 224, 224, 3])
is_channels_first, permute img back
is_channels_first, permute img
torch.Size([8, 3, 224, 224])
torch.Size([8, 224, 224, 3])
is_channels_first, permute img back
{'base_0_rgb': torch.Size([8, 3, 224, 224]), 'left_wrist_0_rgb': torch.Size([8, 3, 224, 224]), 'right_wrist_0_rgb': torch.Size([8, 3, 224, 224])}


In [35]:
output = PI0Pytorch.forward(model, observation, actions)

In [31]:
output.min(), output.max()

(tensor(1.2825e-08, device='cuda:0', grad_fn=<MinBackward1>),
 tensor(179.3414, device='cuda:0', grad_fn=<MaxBackward1>))

# Sampling

In [ ]:
#@title input transform
from openpi.policies import libero_policy
import jax
example = libero_policy.make_libero_example()
transformed_example = model._input_transform(example)
print("example_shape \n", jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, transformed_example))
print("transformed_example shape \n", jax.tree.map(lambda x: x.shape if hasattr(x, "shape") else x, transformed_example))

example_shape 
 {'image': {'base_0_rgb': (224, 224, 3), 'left_wrist_0_rgb': (224, 224, 3), 'right_wrist_0_rgb': (224, 224, 3)}, 'image_mask': {'base_0_rgb': (), 'left_wrist_0_rgb': (), 'right_wrist_0_rgb': ()}, 'state': (32,), 'tokenized_prompt': (48,), 'tokenized_prompt_mask': (48,)}
transformed_example shape 
 {'image': {'base_0_rgb': (224, 224, 3), 'left_wrist_0_rgb': (224, 224, 3), 'right_wrist_0_rgb': (224, 224, 3)}, 'image_mask': {'base_0_rgb': (), 'left_wrist_0_rgb': (), 'right_wrist_0_rgb': ()}, 'state': (32,), 'tokenized_prompt': (48,), 'tokenized_prompt_mask': (48,)}


In [12]:
from openpi.models_pytorch.pi0_pytorch import PI0Pytorch
from openpi.models.model import Observation
observation = Observation.from_dict(
    transformed_example
)
PI0Pytorch.sample_actions(model, device, observation)

TypeCheckError: Type-check error whilst checking the parameters of openpi.models.model.Observation.
The problem arose whilst typechecking parameter 'image_masks'.
Actual value: {'base_0_rgb': True, 'left_wrist_0_rgb': True, 'right_wrist_0_rgb': False}
Expected type: dict[str, Union[Bool[Array, '*b'], Bool[Tensor, '*b'], Bool[ndarray, '*b']]].
----------------------
Called with parameters: {
  'self': Observation(...),
  'images':
  {
    'base_0_rgb': f32[224,224,3](numpy),
    'left_wrist_0_rgb': f32[224,224,3](numpy),
    'right_wrist_0_rgb': f32[224,224,3](numpy)
  },
  'image_masks':
  {'base_0_rgb': True, 'left_wrist_0_rgb': True, 'right_wrist_0_rgb': False},
  'state': f64[32](numpy),
  'tokenized_prompt': i64[48](numpy),
  'tokenized_prompt_mask': bool[48](numpy),
  'token_ar_mask': None,
  'token_loss_mask': None
}
Parameter annotations: (self: Any, images: dict[str, Union[Float[Array, '*b h w c'], Float[Tensor, '*b h w c'], Float[ndarray, '*b h w c']]], image_masks: dict[str, 

In [1]:
time = model.sample_time(bs, device)
action_shape = (bs, 10, 7)
noise = model.sample_noise(action_shape, device)
x_t = noise
suffix_embs, suffix_pad_masks, suffix_att_masks, adarms_cond = model.embed_suffix(state, x_t, time)

NameError: name 'model' is not defined

In [ ]:
suffix_embs, suffix_pad_masks, suffix_att_masks, adarms_cond = model.embed_suffix(state, x_t, time)

In [ ]:
from openpi.models_pytorch.pi0_pytorch import PI0Pytorch, make_att_2d_masks
